In [11]:
import os
import ccxt
import pandas as pd
import ta
import numpy as np
import vectorbt as vbt
import warnings
warnings.filterwarnings('ignore')

In [34]:
df

,Open,High,Low,Close,Volume,signal_engulfing,rsi,signal_rsi,go
Open time,,,,,,,,,
2022-05-10 08:30:00+00:00,31904.00,32016.66,31751.75,31802.67,1730.29420,0,NaN,0,0
2022-05-10 08:45:00+00:00,31802.66,31850.00,31671.59,31749.81,1257.46000,0,NaN,0,0
2022-05-10 09:00:00+00:00,31749.81,31889.74,31703.85,31864.84,1566.35613,0,NaN,0,0
2022-05-10 09:15:00+00:00,31864.85,31932.36,31420.00,31469.90,2241.49614,-1,NaN,0,0
2022-05-10 09:30:00+00:00,31469.91,31687.01,31313.13,31532.29,2189.42806,-1,NaN,0,0
...,...,...,...,...,...,...,...,...,...
2022-10-10 07:15:00+00:00,19418.67,19441.39,19399.83,19412.11,1654.32665,1,44.116970,0,0
2022-10-10 07:30:00+00:00,19413.37,19436.00,19404.03,19423.71,1517.62781,0,46.887346,0,0
2022-10-10 07:45:00+00:00,19423.71,19427.69,19406.20,19412.25,1122.05030,0,44.538237,0,0


In [35]:
df['Close'].iloc[-1]

19213.5

In [22]:
def engulfing_rsi():
    df = vbt.CCXTData.download('BTC/USDT', timeframe='15m', start = '1 day ago' ).get()
    df['signal_engulfing'] = 0
    for current in range(1, len(df.index)):
        previous = current - 1
        if df.iat[current, 3] > df.iat[previous, 0]:
            if df.iat[current, 0] < df.iat[previous, 3]:
                df.iat[current, 5] = 1
        elif df.iat[current, 3] < df.iat[previous, 0]:
            if df.iat[current, 0] > df.iat[previous, 3]:
                df.iat[current, 5] = -1

    df['rsi'] = ta.momentum.RSIIndicator(df['Close']).rsi()
    cond_rsi = [ df['rsi'] < 30, df['rsi'] > 60 ]
    df['signal_rsi'] = np.select(cond_rsi, [1,-1], 0)
    
    cond_go = [ (df['signal_engulfing'] == 1) & (df['signal_rsi'] == 1), 
               (df['signal_engulfing'] == -1) & (df['signal_rsi'] == -1)  ]
    df['go'] =np.select(cond_go, [1, -1], 0)
    
    pf = vbt.Portfolio.from_signals(df['Close'], 
                                entries = (df['go'] == 1), 
                                exits = (df['go'] == -1), 
                                fees = 0.0006,
                                tp_stop = 0.09,
                                sl_stop = 0.03,
                                short_entries = (df['go'] == -1),
                                short_exits = (df['go'] == 1)                                                         
                               )
    
    return df['go'].iloc[-1], df['Close'].iloc[-1]

In [63]:
list_strategy = [engulfing_rsi]
# Secrets come from the environment - never commit them.
#   export TELEGRAM_BOT_TOKEN='123456:ABC...'
#   export TELEGRAM_CHAT_ID='-1234567890'
TELEGRAM_BOT_TOKEN = os.environ['TELEGRAM_BOT_TOKEN']
TELEGRAM_CHAT_ID = os.environ['TELEGRAM_CHAT_ID']
base_url = f'https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendmessage?chat_id={TELEGRAM_CHAT_ID}&text='
def get_signal():
    for strategy in list_strategy:
        (go, closing_price) = strategy()
        if go == 1:
            requests.get(f'{base_url}{strategy.__name__}_LONG @ {closing_price}')
        elif go == -1:
            requests.get(f'{base_url}{strategy.__name__}_SHORT @ {closing_price}')

In [64]:
get_signal()

0it [00:00, ?it/s]

In [65]:
manager = vbt.ScheduleManager()
manager.every(15).minutes.do(get_signal)
manager.start()

0it [00:00, ?it/s]